# The Parquet tables — browse, describe, query

**This is selection data, not measurement.** These tables point at provisions
carrying lexical markers. They say nothing about direction or semantic shift.
Model-run results live elsewhere: in `data/local/runs/` and in the `research`
tables of the DuckDB file, which stay empty until a run exists.

The texts themselves are Swedish statute; only this commentary is English.
Run top to bottom. Change `POOL` in the next cell to see the other pool.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from simulacria.pipeline.silver import POOLS
from simulacria.pipeline.store import TABLE_NAMES, connect

POOL = "v1"  # "v1" = the frozen 50-law corpus, "v2" = the larger pool
TABLES_DIR = POOLS[POOL].tables_dir
connection = connect(TABLES_DIR)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)


def q(sql: str) -> pd.DataFrame:
    """Run SQL against the pool's Parquet files and return a DataFrame."""
    return connection.execute(sql).df()


def show(sql: str, caption: str = "") -> pd.DataFrame:
    """Like q(), but prints the result as well."""
    frame = q(sql)
    if caption:
        print(f"{caption}  ({len(frame)} rows)")
    display(frame)
    return frame


print(f"Pool {POOL}: {TABLES_DIR}")

Pool v1: C:\Users\46762\VSCODE\allegoria\data\local\tables


## 1. Which files exist, and how big are they

Every Parquet file is registered as a view named after the file, so
`select * from candidates` reads `candidates.parquet`.

In [2]:
display(
    pd.DataFrame(
        [
            {
                "table": name,
                "file": f"{name}.parquet",
                "MB": round((TABLES_DIR / f"{name}.parquet").stat().st_size / 1e6, 2),
                "rows": q(f"select count(*) as n from {name}")["n"][0],
                "columns": len(q(f"describe {name}")),
            }
            for name in TABLE_NAMES
        ]
    )
)

,table,file,MB,rows,columns
0,provisions,provisions.parquet,0.40,1952,13
1,candidates,candidates.parquet,0.01,461,12
2,marker_hits,marker_hits.parquet,0.06,8983,5
3,markers,markers.parquet,0.00,28,4


## 2. The columns of each table

`describe` is the shortest path to what a table actually holds. There is no
schema layer to keep in sync — this comes from the file itself.

In [3]:
for name in TABLE_NAMES:
    show(f"describe {name}", f"{name}")

provisions  (13 rows)


,column_name,column_type,null,key,default,extra
0,provision_id,VARCHAR,YES,None,None,None
1,document_id,VARCHAR,YES,None,None,None
2,document_title,VARCHAR,YES,None,None,None
3,document_version,VARCHAR,YES,None,None,None
4,kind,VARCHAR,YES,None,None,None
5,chapter,VARCHAR,YES,None,None,None
6,label,VARCHAR,YES,None,None,None
7,heading,VARCHAR,YES,None,None,None
8,order,INTEGER,YES,None,None,None
9,text,VARCHAR,YES,None,None,None


candidates  (12 rows)


,column_name,column_type,null,key,default,extra
0,provision_id,VARCHAR,YES,None,None,None
1,rank,INTEGER,YES,None,None,None
2,score,DOUBLE,YES,None,None,None
3,has_duty,BOOLEAN,YES,None,None,None
4,has_exception,BOOLEAN,YES,None,None,None
5,has_qualifier,BOOLEAN,YES,None,None,None
6,determinacy,VARCHAR,YES,None,None,None
7,qualifier_determinacy,VARCHAR,YES,None,None,None
8,duty_marker,VARCHAR,YES,None,None,None
9,exception_marker,VARCHAR,YES,None,None,None


marker_hits  (5 rows)


,column_name,column_type,null,key,default,extra
0,provision_id,VARCHAR,YES,None,None,None
1,category,VARCHAR,YES,None,None,None
2,marker,VARCHAR,YES,None,None,None
3,matched_span,VARCHAR,YES,None,None,None
4,char_offset,INTEGER,YES,None,None,None


markers  (4 rows)


,column_name,column_type,null,key,default,extra
0,category,VARCHAR,YES,None,None,None
1,marker,VARCHAR,YES,None,None,None
2,weight,INTEGER,YES,None,None,None
3,pattern,VARCHAR,YES,None,None,None


## 3. The first ten rows of each table

In [4]:
for name in TABLE_NAMES:
    show(f"select * from {name} limit 10", f"{name}, first 10 rows")

provisions, first 10 rows  (10 rows)


,provision_id,document_id,document_title,document_version,kind,chapter,label,heading,order,text,char_count,source_url,source_sha256
0,sfs-1976-580:P1,sfs-1976-580,Lag (1976:580) om medbestämmande i arbetslivet,t.o.m. SFS 2026:886,paragraph,,1 §,Inledande bestämmelser,1,Denna lag äger tillämpning på förhållandet mellan arbetsgivare och arbetstag...,328,https://data.riksdagen.se/dokument/sfs-1976-580.html#P1,4bbf3ba3eb8f659f4685e72d0d9ca1e291fa0e7426764568b91cbe5b023ac006
1,sfs-1976-580:P2,sfs-1976-580,Lag (1976:580) om medbestämmande i arbetslivet,t.o.m. SFS 2026:886,paragraph,,2 §,Inledande bestämmelser,2,"Arbetsgivares verksamhet som är av religiös, vetenskaplig, konstnärlig eller...",268,https://data.riksdagen.se/dokument/sfs-1976-580.html#P2,4bbf3ba3eb8f659f4685e72d0d9ca1e291fa0e7426764568b91cbe5b023ac006
2,sfs-1976-580:P3,sfs-1976-580,Lag (1976:580) om medbestämmande i arbetslivet,t.o.m. SFS 2026:886,paragraph,,3 §,Inledande bestämmelser,3,Innehåller lag eller med stöd av lag meddelad författning särskild föreskrif...,130,https://data.riksdagen.se/dokument/sfs-1976-580.html#P3,4bbf3ba3eb8f659f4685e72d0d9ca1e291fa0e7426764568b91cbe5b023ac006
3,sfs-1976-580:P4,sfs-1976-580,Lag (1976:580) om medbestämmande i arbetslivet,t.o.m. SFS 2026:886,paragraph,,4 §,Inledande bestämmelser,4,Ett avtal är ogiltigt i den mån det innebär att en rättighet eller skyldighe...,1064,https://data.riksdagen.se/dokument/sfs-1976-580.html#P4,4bbf3ba3eb8f659f4685e72d0d9ca1e291fa0e7426764568b91cbe5b023ac006
4,sfs-1976-580:P5,sfs-1976-580,Lag (1976:580) om medbestämmande i arbetslivet,t.o.m. SFS 2026:886,paragraph,,5 §,Inledande bestämmelser,5,Vad i denna lag föreskrivs innebär inte rätt till insyn för en part i sådana...,385,https://data.riksdagen.se/dokument/sfs-1976-580.html#P5,4bbf3ba3eb8f659f4685e72d0d9ca1e291fa0e7426764568b91cbe5b023ac006
5,sfs-1976-580:P6,sfs-1976-580,Lag (1976:580) om medbestämmande i arbetslivet,t.o.m. SFS 2026:886,paragraph,,6 §,Inledande bestämmelser,6,Med arbetstagarorganisation avses sådan sammanslutning av arbetstagare som e...,738,https://data.riksdagen.se/dokument/sfs-1976-580.html#P6,4bbf3ba3eb8f659f4685e72d0d9ca1e291fa0e7426764568b91cbe5b023ac006
6,sfs-1976-580:P7,sfs-1976-580,Lag (1976:580) om medbestämmande i arbetslivet,t.o.m. SFS 2026:886,paragraph,,7 §,Föreningsrätt,7,Med föreningsrätt avses rätt för arbetsgivare och arbetstagare att tillhöra ...,206,https://data.riksdagen.se/dokument/sfs-1976-580.html#P7,4bbf3ba3eb8f659f4685e72d0d9ca1e291fa0e7426764568b91cbe5b023ac006
7,sfs-1976-580:P8,sfs-1976-580,Lag (1976:580) om medbestämmande i arbetslivet,t.o.m. SFS 2026:886,paragraph,,8 §,Föreningsrätt,8,Föreningsrätten skall lämnas okränkt. Kränkning av föreningsrätten föreligge...,891,https://data.riksdagen.se/dokument/sfs-1976-580.html#P8,4bbf3ba3eb8f659f4685e72d0d9ca1e291fa0e7426764568b91cbe5b023ac006
8,sfs-1976-580:P9,sfs-1976-580,Lag (1976:580) om medbestämmande i arbetslivet,t.o.m. SFS 2026:886,paragraph,,9 §,Föreningsrätt,9,"Det åligger arbetsgivar- och arbetstagarorganisation att söka hindra, att me...",226,https://data.riksdagen.se/dokument/sfs-1976-580.html#P9,4bbf3ba3eb8f659f4685e72d0d9ca1e291fa0e7426764568b91cbe5b023ac006
9,sfs-1976-580:P10,sfs-1976-580,Lag (1976:580) om medbestämmande i arbetslivet,t.o.m. SFS 2026:886,paragraph,,10 §,Förhandlingsrätt,10,Arbetstagarorganisation har rätt till förhandling med arbetsgivare i fråga r...,500,https://data.riksdagen.se/dokument/sfs-1976-580.html#P10,4bbf3ba3eb8f659f4685e72d0d9ca1e291fa0e7426764568b91cbe5b023ac006


candidates, first 10 rows  (10 rows)


,provision_id,rank,score,has_duty,has_exception,has_qualifier,determinacy,qualifier_determinacy,duty_marker,exception_marker,qualifier_marker,char_count
0,sfs-2026-585:P18,1,21.0,True,True,True,specific,unmarked,ska,trots,endast om,732
1,sfs-2026-1054:P8,2,19.0,True,True,True,specific,unmarked,ska,om inte,i den mån,541
2,sfs-1977-480:P11,3,19.0,True,True,True,specific,vague,ska,dock,särskilda skäl,548
3,sfs-2026-1283:K5P1,4,19.0,True,True,True,specific,unmarked,ska,dock,i den mån,549
4,sfs-1982-80:P10,5,19.0,True,True,True,specific,unmarked,ska,om inte,om,555
5,sfs-1982-80:P20,6,19.0,True,True,True,specific,unmarked,ska,om inte,om,561
6,sfs-1977-480:P19,7,19.0,True,True,True,specific,unmarked,ska,dock,endast om,613
7,sfs-1982-80:P18,8,19.0,True,True,True,specific,unmarked,får inte,dock,särskilda skäl,675
8,sfs-2026-786:K5P2,9,19.0,True,True,True,specific,unmarked,ska,i stället,endast om,856
9,sfs-2026-408:K9P10,10,19.0,True,True,True,specific,unmarked,får,gäller inte,endast om,882


marker_hits, first 10 rows  (10 rows)


,provision_id,category,marker,matched_span,char_offset
0,sfs-1976-580:P1,duty,ska,skall,285
1,sfs-1976-580:P2,exception,undantag,undantages,180
2,sfs-1976-580:P2,qualifier,i den mån,såvitt,222
3,sfs-1976-580:P4,duty,ska,ska,376
4,sfs-1976-580:P4,duty,får inte,får inte,329
5,sfs-1976-580:P4,duty,får,får,146
6,sfs-1976-580:P4,duty,får,får,329
7,sfs-1976-580:P4,duty,får,får,878
8,sfs-1976-580:P4,exception,dock,dock,154
9,sfs-1976-580:P4,qualifier,i den mån,i den mån,22


markers, first 10 rows  (10 rows)


,category,marker,weight,pattern
0,duty,ska,2,\bska(?:ll)?\b
1,duty,får inte,2,"\bfår\s+(?:\w+\s+){0,3}?inte\b"
2,duty,måste,2,\bmåste\b
3,duty,är skyldig,2,\b(?:är|blir)\s+skyldiga?\b|\båligger\b
4,duty,förbjuden,2,\bförbjud(?:en|et|na)\b|\bförbud\b
5,duty,bör,1,\bbör\b
6,duty,får,1,\bfår\b
7,duty,har rätt att,1,\b(?:har|äger)\s+rätt\s+(?:att|till)\b
8,exception,dock,3,\bdock\b
9,exception,om inte,3,"\bom\s+(?:\w+\s+){0,3}?inte\b"


## 4. The ten highest-ranked candidates

`candidates` are ranked provisions. `score` is the sum of marker weights, and
`determinacy` says whether the duty marker is specific or unmarked. This is a
selection measure: a high score means many markers, not clear normativity.

In [5]:
show(
    """
    select c.rank, c.provision_id, c.score, c.determinacy,
           c.duty_marker, c.exception_marker, c.qualifier_marker,
           p.document_title, substr(p.text, 1, 120) as text_start
    from candidates c
    join provisions p on p.provision_id = c.provision_id
    order by c.rank
    limit 10
    """,
    "Top 10 candidates",
)

Top 10 candidates  (10 rows)


,rank,provision_id,score,determinacy,duty_marker,exception_marker,qualifier_marker,document_title,text_start
0,1,sfs-2026-585:P18,21.0,specific,ska,trots,endast om,Lag (2026:585) om regeringens godkännande av kärntekniska anläggningar,Regeringen får godkänna en kärnteknisk anläggning och besluta om en anläggni...
1,2,sfs-2026-1054:P8,19.0,specific,ska,om inte,i den mån,Lag (2026:1054) om karens för vissa befattningshavare i Finansinspektionen v...,Ett beslut om karens och ersättning ska fattas av\n\n1. Nämnden för prövning...
2,3,sfs-1977-480:P11,19.0,specific,ska,dock,särskilda skäl,Semesterlag (1977:480),"Om det inte går att komma överens om hur semesterledighet ska förläggas, bes..."
3,4,sfs-2026-1283:K5P1,19.0,specific,ska,dock,i den mån,Lag (2026:1283) om elektriska ledningar,En nätkoncession för linje som gäller tills vidare får omprövas i fråga om l...
4,5,sfs-1982-80:P10,19.0,specific,ska,om inte,om,Lag (1982:80) om anställningsskydd,Uppsägningsbeskedet skall lämnas till arbetstagaren personligen. Är det inte...
5,6,sfs-1982-80:P20,19.0,specific,ska,om inte,om,Lag (1982:80) om anställningsskydd,Beskedet om avskedande skall lämnas till arbetstagaren personligen.\n\nÄr de...
6,7,sfs-1977-480:P19,19.0,specific,ska,dock,endast om,Semesterlag (1977:480),Vill en arbetstagare spara semesterdagar eller ta ut sparade semesterdagar i...
7,8,sfs-1982-80:P18,19.0,specific,får inte,dock,särskilda skäl,Lag (1982:80) om anställningsskydd,"Avskedande får ske, om arbetstagaren grovt har åsidosatt sina åligganden mot..."
8,9,sfs-2026-786:K5P2,19.0,specific,ska,i stället,endast om,Lag (2026:786) om nordisk verkställighet i brottmål,En dömd person som befinner sig i Sverige får på begäran av en behörig myndi...
9,10,sfs-2026-408:K9P10,19.0,specific,får,gäller inte,endast om,Vapenlag (2026:408),En vapenhandlare får under högst två veckor låna ut ett skjutvapen till den ...


,rank,provision_id,score,determinacy,duty_marker,exception_marker,qualifier_marker,document_title,text_start
0,1,sfs-2026-585:P18,21.0,specific,ska,trots,endast om,Lag (2026:585) om regeringens godkännande av kärntekniska anläggningar,Regeringen får godkänna en kärnteknisk anläggning och besluta om en anläggni...
1,2,sfs-2026-1054:P8,19.0,specific,ska,om inte,i den mån,Lag (2026:1054) om karens för vissa befattningshavare i Finansinspektionen v...,Ett beslut om karens och ersättning ska fattas av\n\n1. Nämnden för prövning...
2,3,sfs-1977-480:P11,19.0,specific,ska,dock,särskilda skäl,Semesterlag (1977:480),"Om det inte går att komma överens om hur semesterledighet ska förläggas, bes..."
3,4,sfs-2026-1283:K5P1,19.0,specific,ska,dock,i den mån,Lag (2026:1283) om elektriska ledningar,En nätkoncession för linje som gäller tills vidare får omprövas i fråga om l...
4,5,sfs-1982-80:P10,19.0,specific,ska,om inte,om,Lag (1982:80) om anställningsskydd,Uppsägningsbeskedet skall lämnas till arbetstagaren personligen. Är det inte...
5,6,sfs-1982-80:P20,19.0,specific,ska,om inte,om,Lag (1982:80) om anställningsskydd,Beskedet om avskedande skall lämnas till arbetstagaren personligen.\n\nÄr de...
6,7,sfs-1977-480:P19,19.0,specific,ska,dock,endast om,Semesterlag (1977:480),Vill en arbetstagare spara semesterdagar eller ta ut sparade semesterdagar i...
7,8,sfs-1982-80:P18,19.0,specific,får inte,dock,särskilda skäl,Lag (1982:80) om anställningsskydd,"Avskedande får ske, om arbetstagaren grovt har åsidosatt sina åligganden mot..."
8,9,sfs-2026-786:K5P2,19.0,specific,ska,i stället,endast om,Lag (2026:786) om nordisk verkställighet i brottmål,En dömd person som befinner sig i Sverige får på begäran av en behörig myndi...
9,10,sfs-2026-408:K9P10,19.0,specific,får,gäller inte,endast om,Vapenlag (2026:408),En vapenhandlare får under högst två veckor låna ut ett skjutvapen till den ...


## 5. Distributions — what the selection is made of

In [6]:
show(
    """
    select determinacy, count(*) as candidates,
           round(avg(score), 1) as mean_score,
           sum(case when has_exception then 1 else 0 end) as with_exception
    from candidates group by 1 order by candidates desc
    """,
    "Candidates per determinacy",
)
show(
    """
    select m.category, m.marker, m.weight, count(h.provision_id) as hits,
           count(distinct h.provision_id) as provisions
    from markers m
    left join marker_hits h on h.marker = m.marker and h.category = m.category
    group by 1, 2, 3 order by hits desc limit 15
    """,
    "The 15 most frequent markers",
)

Candidates per determinacy  (3 rows)


,determinacy,candidates,mean_score,with_exception
0,unmarked,298,10.8,298.0
1,specific,119,15.3,119.0
2,vague,44,12.0,44.0


The 15 most frequent markers  (15 rows)


,category,marker,weight,hits,provisions
0,qualifier,om,1,3423,1449
1,duty,ska,2,2076,1134
2,duty,får,1,1195,830
3,qualifier,när/vid,1,359,291
4,exception,om inte,3,275,232
5,duty,får inte,2,211,189
6,exception,dock,3,206,191
7,qualifier,i den mån,2,178,154
8,qualifier,efter det att,1,158,129
9,exception,gäller inte,2,147,132


,category,marker,weight,hits,provisions
0,qualifier,om,1,3423,1449
1,duty,ska,2,2076,1134
2,duty,får,1,1195,830
3,qualifier,när/vid,1,359,291
4,exception,om inte,3,275,232
5,duty,får inte,2,211,189
6,exception,dock,3,206,191
7,qualifier,i den mån,2,178,154
8,qualifier,efter det att,1,158,129
9,exception,gäller inte,2,147,132


## 6. Look up a single provision

Replace `PROVISION_ID` with any id from the tables above.

In [7]:
PROVISION_ID = q("select provision_id from candidates order by rank limit 1")["provision_id"][0]
display(q(f"select * from provisions where provision_id = '{PROVISION_ID}'").T)
show(
    f"""
    select category, marker, matched_span, char_offset from marker_hits
    where provision_id = '{PROVISION_ID}' order by char_offset
    """,
    f"Marker hits in {PROVISION_ID}",
)
print(q(f"select text from provisions where provision_id = '{PROVISION_ID}'")["text"][0])

,0
provision_id,sfs-2026-585:P18
document_id,sfs-2026-585
document_title,Lag (2026:585) om regeringens godkännande av kärntekniska anläggningar
document_version,None
kind,paragraph
chapter,
label,18 §
heading,Kommunens tillstyrkande
order,18
text,Regeringen får godkänna en kärnteknisk anläggning och besluta om en anläggni...


Marker hits in sfs-2026-585:P18  (10 rows)


,category,marker,matched_span,char_offset
0,duty,får,får,11
1,qualifier,om,om,62
2,qualifier,endast om,endast om,85
3,qualifier,om,om,92
4,duty,ska,ska,148
5,qualifier,i den mån,I fråga om,250
6,qualifier,om,om,258
7,duty,får,får,326
8,exception,trots,trots att,376
9,qualifier,om,om,438


Regeringen får godkänna en kärnteknisk anläggning och besluta om en anläggningsplan, endast om den eller de kommuner som ligger inom de områden som ska omfattas av planen har tillstyrkt ansökan respektive planen.

Ett tillstyrkande gäller i fem år.

I fråga om mellanlagring eller slutförvaring av kärnämnen eller kärn-avfall får regeringen fatta beslut enligt första stycket trots att en kommun inte har tillstyrkt ansökan eller planen, om det från nationell synpunkt är synnerligen angeläget att anläggningen uppförs och drivs och det inte finns någon annan plats

1. som bedöms vara lämpligare för anläggningen, eller

2. som är lämplig och har anvisats för anläggningen inom en annan kommun som kan antas godta en placering där.


## 7. Full-text search in the statute text

The search string is Swedish because the corpus is: `endast om` is "only if".

In [8]:
show(
    """
    select provision_id, document_title, substr(text, 1, 150) as extract
    from provisions where text ilike '%endast om%' limit 10
    """,
    "Provisions containing 'endast om'",
)

Provisions containing 'endast om'  (10 rows)


,provision_id,document_title,extract
0,sfs-1976-580:P39,Lag (1976:580) om medbestämmande i arbetslivet,Har förhandling enligt 38 § ägt rum och förklarar den centrala arbetstagaror...
1,sfs-1977-480:P13,Semesterlag (1977:480),Beslutar en arbetsgivare om permittering som arbetstagaren inte hade anledni...
2,sfs-1977-480:P19,Semesterlag (1977:480),Vill en arbetstagare spara semesterdagar eller ta ut sparade semesterdagar i...
3,sfs-1977-480:P30b,Semesterlag (1977:480),Om det innan en anställning har upphört står klart att en ny anställning mel...
4,sfs-1982-673:P3,Arbetstidslag (1982:673),Genom ett kollektivavtal som har slutits eller godkänts av en central arbets...
5,sfs-1995-584:P15b,Föräldraledighetslag (1995:584),Om en förälder med barn som inte har fyllt åtta år begär flexibla arbetsform...
6,sfs-1995-584:P20,Föräldraledighetslag (1995:584),Rätt till omplacering enligt 18 och 19 §§ gäller endast om det skäligen kan ...
7,sfs-2026-1011:K3P10,Konsumentkreditlag (2026:1011),En kreditförmedlare ska informera konsumenten om avgifter som konsumenten sk...
8,sfs-2026-1011:K3P13,Konsumentkreditlag (2026:1011),En näringsidkare ska pröva om konsumenten har ekonomiska förutsättningar att...
9,sfs-2026-1011:K3P18,Konsumentkreditlag (2026:1011),En kreditgivare som erbjuder försäkring för beviljade krediter ska säkerstäl...


,provision_id,document_title,extract
0,sfs-1976-580:P39,Lag (1976:580) om medbestämmande i arbetslivet,Har förhandling enligt 38 § ägt rum och förklarar den centrala arbetstagaror...
1,sfs-1977-480:P13,Semesterlag (1977:480),Beslutar en arbetsgivare om permittering som arbetstagaren inte hade anledni...
2,sfs-1977-480:P19,Semesterlag (1977:480),Vill en arbetstagare spara semesterdagar eller ta ut sparade semesterdagar i...
3,sfs-1977-480:P30b,Semesterlag (1977:480),Om det innan en anställning har upphört står klart att en ny anställning mel...
4,sfs-1982-673:P3,Arbetstidslag (1982:673),Genom ett kollektivavtal som har slutits eller godkänts av en central arbets...
5,sfs-1995-584:P15b,Föräldraledighetslag (1995:584),Om en förälder med barn som inte har fyllt åtta år begär flexibla arbetsform...
6,sfs-1995-584:P20,Föräldraledighetslag (1995:584),Rätt till omplacering enligt 18 och 19 §§ gäller endast om det skäligen kan ...
7,sfs-2026-1011:K3P10,Konsumentkreditlag (2026:1011),En kreditförmedlare ska informera konsumenten om avgifter som konsumenten sk...
8,sfs-2026-1011:K3P13,Konsumentkreditlag (2026:1011),En näringsidkare ska pröva om konsumenten har ekonomiska förutsättningar att...
9,sfs-2026-1011:K3P18,Konsumentkreditlag (2026:1011),En kreditgivare som erbjuder försäkring för beviljade krediter ska säkerstäl...


## 8. Your own queries, and CSV export

Write anything in a new cell: `show("select ... from provisions")`.

To open a table in Excel, call `to_csv(...)` in a cell of your own. The file
lands under `data/local/`, which git ignores. The semicolon delimiter makes
Swedish Excel split the columns without an import dialog.

In [9]:
def to_csv(sql: str, path: Path) -> Path:
    """Write a query result straight to CSV through DuckDB, bypassing pandas."""
    target = str(path.resolve()).replace("'", "''")
    connection.execute(f"copy ({sql}) to '{target}' (header, delimiter ';')")
    return path


# to_csv("select * from candidates", ROOT / "data/local/candidates.csv")

## 9. The other database: model-run results

`data/local/allegoria.duckdb` holds the same selection tables under the
`selection` schema, plus the `research` tables for model runs. Rebuild it with
`python scripts/pipeline/build_database.py`; it stays empty of runs until one exists.
The view to go to then is `research.slot_comparisons`, which lines up a source
slot with the parent and child readings of it.

In [10]:
import duckdb

DATABASE = ROOT / "data/local/allegoria.duckdb"
if DATABASE.is_file():
    with duckdb.connect(str(DATABASE), read_only=True) as run_database:
        tables = run_database.execute(
            "select table_schema, table_name from information_schema.tables "
            "where table_schema in ('research', 'selection') order by 1, 2"
        ).fetchall()
        display(
            pd.DataFrame(
                [
                    {
                        "schema": schema,
                        "table": table,
                        "rows": run_database.execute(
                            f'select count(*) from "{schema}"."{table}"'
                        ).fetchone()[0],
                    }
                    for schema, table in tables
                ]
            )
        )
else:
    print(f"{DATABASE} is missing — run python scripts/pipeline/build_database.py")

,schema,table,rows
0,research,call_events,0
1,research,readings,0
2,research,runs,0
3,research,slot_comparisons,0
4,research,slot_observations,0
5,research,source_slots,0
6,research,texts,0
7,selection,candidates,461
8,selection,marker_hits,8983
9,selection,markers,28
